# W1-12 / EEC-16: frozen-encoder acoustic classifier (GPU)
Trains a 20-way linear head on cached, frozen AraBEST-RQ-300M-6k features.
See `docs/acoustic_features.md` and `configs/acoustic_classifier.json`.

**Before running:** Runtime → Change runtime type → **T4 GPU** (or better) → Save. Then run the cells top to bottom.

`MODE` in the first cell picks the step:
- `'overfit_check'`: extract features for all 700 pilot clips into the feature cache on Drive, then check
  that the head can memorise a 40-clip training subset. Stops before pilot training.
- `'full'`: reuse the cached features, repeat the overfit check, train on the 500-clip training pilot,
  predict the 200-clip dev pilot with all 20 logits, and run the shared evaluator.

The feature cache lives on Drive, so the `'full'` run in a later session reuses the same features instead
of recomputing them. The 1.24 GB checkpoint downloads to Colab's local disk: Drive corrupted it in an earlier run.
The dev pilot is in-source ADI-20, not out-of-domain. Each run writes to a new folder; nothing is overwritten.

In [ ]:
from google.colab import drive
from pathlib import Path
import os

drive.mount('/content/drive')
MODE = 'overfit_check'  # 'overfit_check' (milestone 2) or 'full'
# Same data folder as the W1-11 notebook, so pilot WAVs already fetched there are reused.
DATA_ROOT = Path('/content/drive/MyDrive/Arabic-Dialect-Identification-data')
AUDIO_ROOT = DATA_ROOT / 'audio/adi20_micro/c8da454c9f474bb075d3fe608940ee29e8ccfa43'
OUTPUT_ROOT = DATA_ROOT / 'output'  # reusable feature cache: OUTPUT_ROOT/features/...
CACHE_ROOT = Path('/content/hf-cache')  # checkpoint download, local disk only
for folder in (DATA_ROOT, OUTPUT_ROOT, CACHE_ROOT):
    folder.mkdir(parents=True, exist_ok=True)
assert MODE in ('overfit_check', 'full'), MODE
print('Mode:', MODE)
print('Audio folder:', AUDIO_ROOT)
print('Feature cache root:', OUTPUT_ROOT)

## Get the code
The default `SOURCE = 'github'` fetches the reviewed immutable commit.
Optional `SOURCE = 'zip'` uploads `Arabic-Dialect-Identification-w1-12.zip` built from a local checkout, for code that is
not pushed yet. Build it from the repository root with:

    (git ls-files -co --exclude-standard; find .git -type f) | zip -q Arabic-Dialect-Identification-w1-12.zip -@

The zip includes `.git`, so the run record still names the base commit and marks uncommitted changes as dirty.
`SOURCE = 'github'` fetches `PINNED_COMMIT` once the reviewed code is pushed.

In [ ]:
import hashlib, json, shutil, subprocess, sys, zipfile
from datetime import datetime, timezone

SOURCE = 'github'  # or 'zip' for a local development checkout
REPOSITORY_URL = 'https://github.com/alizbeeb5/Arabic-Dialect-Identification.git'
PINNED_COMMIT = '5f37987867d9eeaf52657b72ce7de334ebd3c11a'  # reviewed extraction fixes
ZIP_NAME = 'Arabic-Dialect-Identification-w1-12.zip'
REPO_ROOT = Path('/content/Arabic-Dialect-Identification')

def git(*args):
    return subprocess.run(['git', '-C', str(REPO_ROOT), *args], check=True, capture_output=True, text=True).stdout.strip()

shutil.rmtree(REPO_ROOT, ignore_errors=True)
if SOURCE == 'zip':
    from google.colab import files
    uploaded = files.upload()  # choose Arabic-Dialect-Identification-w1-12.zip
    if ZIP_NAME not in uploaded:
        raise RuntimeError(f'Upload {ZIP_NAME}; got {sorted(uploaded)}')
    SOURCE_SHA256 = hashlib.sha256(uploaded[ZIP_NAME]).hexdigest()
    with zipfile.ZipFile(ZIP_NAME) as archive:
        archive.extractall(REPO_ROOT)
elif SOURCE == 'github':
    if not (isinstance(PINNED_COMMIT, str) and len(PINNED_COMMIT) == 40):
        raise RuntimeError('Set PINNED_COMMIT to the reviewed commit before using SOURCE = github')
    REPO_ROOT.mkdir(parents=True)
    git('init', '--quiet')
    git('remote', 'add', 'origin', REPOSITORY_URL)
    git('fetch', '--quiet', '--depth', '1', 'origin', PINNED_COMMIT)
    git('checkout', '--quiet', '--detach', 'FETCH_HEAD')
    SOURCE_SHA256 = None
else:
    raise ValueError(f'Unknown SOURCE {SOURCE!r}')

SOURCE_COMMIT = git('rev-parse', 'HEAD')
SOURCE_STATUS = git('status', '--porcelain')
if SOURCE == 'github' and SOURCE_COMMIT != PINNED_COMMIT:
    raise RuntimeError(f'Source mismatch: expected {PINNED_COMMIT}, got {SOURCE_COMMIT}')
if not (REPO_ROOT / 'src/experiments/acoustic_classifier.py').is_file():
    raise RuntimeError('This source does not contain src/experiments/acoustic_classifier.py')

RUN_ROOT = DATA_ROOT / 'runs/acoustic_classifier_w1_12' / (datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ') + f'_{MODE}')
print('Base commit:', SOURCE_COMMIT)
print('Uncommitted changes:', SOURCE_STATUS or 'none')
print('Output folder:', RUN_ROOT)

## Install the acoustic environment, check the GPU and run the offline tests

In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(REPO_ROOT / 'environments/acoustic.txt')], check=True)
probe = subprocess.run([sys.executable, '-c',
    'import torch, speechbrain; print(torch.__version__, speechbrain.__version__, torch.cuda.is_available(), '
    'torch.cuda.get_device_name(0) if torch.cuda.is_available() else None)'],
    capture_output=True, text=True, check=True).stdout.strip()
print('torch, speechbrain, CUDA, GPU:', probe)
assert ' True ' in f' {probe} ', 'No GPU: Runtime -> Change runtime type -> T4 GPU, then rerun from the top'
for pattern in ('test_acoustic_encoder.py', 'test_acoustic_classifier.py'):
    result = subprocess.run([sys.executable, '-m', 'unittest', 'discover', '-s', 'tests', '-p', pattern],
                            cwd=REPO_ROOT, capture_output=True, text=True)
    print(pattern, result.stderr.strip().splitlines()[-1])
    assert result.returncode == 0, result.stderr

## Make sure the 700 pilot WAVs are on Drive
Fetches only missing pilot clips (about 0.4 GB in total) from the pinned public release.

In [ ]:
# Presence check; the experiment verifies every WAV SHA256 before extracting it.
def missing_pilot_clips():
    missing = []
    for split in ('train', 'dev'):
        rows = [json.loads(l) for l in (REPO_ROOT / f'data/manifests/{split}_pilot.jsonl').read_text().splitlines() if l.strip()]
        missing += [r['clip_id'] for r in rows if not (AUDIO_ROOT / r['audio_path']).is_file()]
    return missing

missing = missing_pilot_clips()
print(f'{700 - len(missing)} of 700 pilot WAV files already present.')
if missing:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(REPO_ROOT / 'requirements-data.txt')], check=True)
    result = subprocess.run([sys.executable, '-u', str(REPO_ROOT / 'src/data/fetch_pilot_audio.py'),
                             '--audio-root', str(AUDIO_ROOT)], cwd=REPO_ROOT)
    assert result.returncode == 0, 'Some clips could not be fetched or verified; see the messages above.'
    missing = missing_pilot_clips()
assert not missing, f'{len(missing)} pilot WAV files are still missing, e.g. {missing[:3]}'
print('All 700 pilot WAV files present.')

## Run
Exit 0 = no clip failures; exit 2 = finished with explicit clip failures (kept in the record).
If the overfit check fails, the run stops with exit 1 and `overfit_check.json` / `run_record.json` explain why.

The config extracts one clip at a time (`batch_size: 1`): every clip is encoded alone with no padding, as the
official model would. Attention memory grows with the square of the padded length, so a batch of 8 dev clips
padded to the 142 s clip needed over 17 GB and ran out of memory on a T4; one 142 s clip alone needs about 1 GB.
If an earlier run cached features with a different batch size, delete `OUTPUT_ROOT / 'features'` first so all
700 clips are extracted in one run with one setting (batch size is not part of the cache key).

In [ ]:
command = [sys.executable, '-u', str(REPO_ROOT / 'src/experiments/acoustic_classifier.py'),
           '--audio-root', str(AUDIO_ROOT), '--output-root', str(OUTPUT_ROOT),
           '--cache-root', str(CACHE_ROOT), '--device', 'cuda', '--output-dir', str(RUN_ROOT)]
if MODE == 'overfit_check':
    command.append('--overfit-check-only')
print(' '.join(command))
process = subprocess.Popen(command, cwd=REPO_ROOT, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
log_lines = []
for line in process.stdout:
    print(line, end='', flush=True)
    log_lines.append(line)
exit_code = process.wait()
if RUN_ROOT.is_dir():
    (RUN_ROOT / 'console.log').write_text(''.join(log_lines), encoding='utf-8')
print('Exit code:', exit_code)

## Summary to send back

In [ ]:
record_path = RUN_ROOT / 'run_record.json'
assert record_path.is_file(), f'No run record; the run stopped before the overfit check (exit {exit_code})'
record = json.loads(record_path.read_text())
record['code']['notebook_source'] = {'kind': SOURCE, 'commit': SOURCE_COMMIT, 'uncommitted': SOURCE_STATUS,
                                     'zip_sha256': SOURCE_SHA256}
record_path.write_text(json.dumps(record, indent=2, ensure_ascii=False) + '\n')

frozen, feats, res, gate = record['encoder']['frozen'], record['features'], record['resources'], record['overfit_check']
print('Status:', record['status'], '| exit code:', exit_code)
print(f"Encoder {record['encoder']['revision'][:12]}: {frozen['trainable_parameters']} trainable of "
      f"{frozen['parameters']:,} parameters, weights unchanged: {frozen['state_sha256_before'] == frozen['state_sha256_after']}")
print('Extracted this run:', feats['extracted_this_run'], '| reused from cache:', feats['reused_from_cache'])
gb = None if res['peak_gpu_memory_allocated_bytes'] is None else round(res['peak_gpu_memory_allocated_bytes'] / 2**30, 2)
print(f"Extraction {res['feature_extraction_sec']:.0f}s, {res['extraction_audio_sec_per_sec']} audio-s/s, "
      f"peak GPU {gb} GB on {record['environment']['gpu']}")
for split in ('train', 'dev'):
    s = feats[f'{split}_stats']
    print(f"{split} features: mean {s['mean']:.3f}, mean|x| {s['mean_abs']:.3f}, max|x| {s['max_abs']:.2f}, "
          f"dim std {s['mean_dim_std']:.3f}, constant dims {s['constant_dims']}, duplicates {s['duplicate_rows']}, "
          f"mean cosine {s['mean_offdiag_cosine']:.3f}")
print(f"Overfit check: loss {gate['initial_loss']:.4f} -> {gate['final_loss']:.4f} (chance {gate['chance_loss']:.4f}), "
      f"accuracy {gate['initial_accuracy']:.2%} -> {gate['final_accuracy']:.2%}, passed: {gate['passed']}")
if record.get('development_evaluation'):
    final, metrics = record['final_epoch'], json.loads((RUN_ROOT / 'evaluation/metrics.json').read_text())
    print(f"Epoch {final['epoch']}: train loss {final['train']['loss']:.4f} acc {final['train']['accuracy']:.2%} | "
          f"dev loss {final['dev']['loss']:.4f} acc {final['dev']['accuracy']:.2%}")
    print(f"Dev accuracy {metrics['accuracy']:.2%} | macro-F1 {metrics['macro_f1']:.4f} | coverage {metrics['coverage']:.0%}")
    print('Model revision:', record['model_revision'])

## Hand the result back
Zips the run folder (run record, overfit check, console log and, for `'full'`, checkpoint, predictions,
learning curve and metrics) and downloads it. The feature cache stays on Drive for the next run.

In [ ]:
from google.colab import files
archive = shutil.make_archive(f'/content/acoustic_classifier_w1_12_{RUN_ROOT.name}', 'zip', RUN_ROOT)
print('Saved on Drive:', RUN_ROOT)
files.download(archive)